# 03 Explainability and Evaluation

Final-day notebook for Person C. Run Person A preprocessing, Person A trust pipeline, and Person B model verification before running explainability outputs.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath('..'))

import pandas as pd
from shared import config
from evaluation.model_evaluation import (
    evaluate_predictions,
    save_metrics,
    save_classification_report,
    save_predictions,
    save_confusion_matrix_plot,
)
from evaluation.fairness_metrics import (
    evaluate_fairness,
    prediction_distribution,
    save_fairness_results,
    save_prediction_distribution,
)
from explainability.shap_explainer import load_label_mapping, split_correct_and_misclassified

print('Imports ready')

In [ ]:
required_files = [
    config.PROCESSED_HATE_SPEECH_PATH,
    config.PROCESSED_HATEXPLAIN_PATH,
    config.TRAINING_DATASET_PATH,
    config.PROCESSED_ANNOTATOR_WEIGHTS_PATH,
    config.PROCESSED_SOFT_LABELS_PATH,
]
model_files = [
    config.TRAINED_MODEL_PATH if config.TRAINED_MODEL_PATH.exists() else config.ROOT_TRAINED_MODEL_PATH,
    config.LABEL_MAPPING_PATH if config.LABEL_MAPPING_PATH.exists() else config.ROOT_LABEL_MAPPING_PATH,
]

for path in required_files + model_files:
    print(f'{path}:', 'FOUND' if Path(path).exists() else 'MISSING')

In [ ]:
dataset_path = config.FINAL_TRAINING_DATASET_PATH if config.FINAL_TRAINING_DATASET_PATH.exists() else config.TRAINING_DATASET_PATH
training_df = pd.read_csv(dataset_path)
print('Training dataset loaded:', training_df.shape)
training_df.head()

## Model Evaluation

If `model_prediction` is missing, the notebook attempts to load Person B's saved model and generate predictions before saving metrics.

In [ ]:
def run_model_inference(dataframe):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoModel, AutoTokenizer
    from demographic_module.trust_embedding import TrustEmbedding
    from demographic_module.fusion_layer import FusionLayer

    class DemographicModernBERT(nn.Module):
        def __init__(self, num_labels):
            super().__init__()
            self.bert = AutoModel.from_pretrained(config.MODEL_NAME)
            self.trust_embedding = TrustEmbedding()
            self.fusion_layer = FusionLayer(
                bert_hidden_size=self.bert.config.hidden_size,
                demographic_size=64,
                output_size=self.bert.config.hidden_size,
            )
            self.dropout = nn.Dropout(0.2)
            self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

        def forward(self, input_ids, attention_mask, confidence, p_normal, p_offensive, p_hate):
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_embedding = outputs.last_hidden_state[:, 0]
            trust_vector = self.trust_embedding(confidence, p_normal, p_offensive, p_hate)
            fused_embedding = self.fusion_layer(cls_embedding, trust_vector)
            return self.classifier(self.dropout(fused_embedding))

    class ExplainabilityDataset(Dataset):
        def __init__(self, frame, tokenizer):
            self.frame = frame.reset_index(drop=True)
            self.tokenizer = tokenizer

        def __len__(self):
            return len(self.frame)

        def __getitem__(self, index):
            row = self.frame.iloc[index]
            text = str(row.get('clean_text', row.get('text', '')))
            encoding = self.tokenizer(
                text,
                truncation=True,
                padding='max_length',
                max_length=config.MAX_LENGTH,
                return_tensors='pt',
            )
            return {
                'input_ids': encoding['input_ids'].squeeze(0),
                'attention_mask': encoding['attention_mask'].squeeze(0),
                'confidence': torch.tensor(row.get('confidence', 1.0), dtype=torch.float),
                'p_normal': torch.tensor(row.get('p_normal', 0.0), dtype=torch.float),
                'p_offensive': torch.tensor(row.get('p_offensive', 0.0), dtype=torch.float),
                'p_hate': torch.tensor(row.get('p_hate', 0.0), dtype=torch.float),
            }

    model_path = config.TRAINED_MODEL_PATH if config.TRAINED_MODEL_PATH.exists() else config.ROOT_TRAINED_MODEL_PATH
    if not model_path.exists():
        raise FileNotFoundError(f'Trained model not found: {model_path}')

    label_mapping = load_label_mapping()
    inverse_mapping = {idx: label for label, idx in label_mapping.items()}
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
    model = DemographicModernBERT(num_labels=len(label_mapping))
    checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, dict):
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        model.load_state_dict(state_dict)
    else:
        model = checkpoint

    model.to(device)
    model.eval()
    loader = DataLoader(ExplainabilityDataset(dataframe, tokenizer), batch_size=config.BATCH_SIZE)
    predictions = []

    with torch.no_grad():
        for batch in loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(**batch)
            predictions.extend(torch.argmax(logits, dim=1).cpu().tolist())

    return [inverse_mapping[prediction] for prediction in predictions]


if 'model_prediction' not in training_df.columns:
    try:
        training_df['model_prediction'] = run_model_inference(training_df)
        print('Model inference completed')
    except Exception as exc:
        print('Model inference skipped:', exc)
        training_df['model_prediction'] = training_df['predicted_label']
        print('Using predicted_label as fallback for notebook verification')

In [ ]:
true_col = 'predicted_label'
pred_col = 'model_prediction' if 'model_prediction' in training_df.columns else 'predicted_label'

metrics, report, matrix = evaluate_predictions(training_df, true_col, pred_col)
labels = sorted(training_df[true_col].dropna().unique())

save_metrics(metrics)
save_classification_report(training_df[true_col], training_df[pred_col])
save_predictions(training_df, training_df[true_col], training_df[pred_col])
save_confusion_matrix_plot(matrix, labels)

metrics

## Fairness Evaluation

Fairness metrics are calculated when protected attribute columns are available in the dataset.

In [ ]:
fairness_results = evaluate_fairness(training_df, true_col, pred_col)
distribution = prediction_distribution(training_df, pred_col)

save_fairness_results(fairness_results)
save_prediction_distribution(distribution)

fairness_results.head()

## Correct and Misclassified Samples

These subsets can be passed into SHAP and Captum once the trained model and inference function are available.

In [ ]:
predictions_df = training_df.copy()
predictions_df['true_label'] = training_df[true_col]
predictions_df['predicted_label'] = training_df[pred_col]
correct_samples, misclassified_samples = split_correct_and_misclassified(predictions_df)

print('Correct samples:', len(correct_samples))
print('Misclassified samples:', len(misclassified_samples))